# Data pipeline executor

Single control panel for running both data pipelines in this project:

- **`data_gen/`** — synthetic generator (lines / pages+YOLO labels / PDFs).
- **`real_data/`** — real external Hugging Face OCR data, sliced into
  overlapping chunks for Conformer-encoder input windowing.

Each stage below is a thin wrapper around the same CLI entry points documented
in `data_gen/README.md` / `real_data/README.md` (`python -m ...`), run as a
subprocess so this notebook never drifts from what those scripts actually do.
Edit `CONFIG` in the next code cell, then run whichever stage cells you need
— they're independent, run them in any order/subset.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    from pathlib import Path

    # This repo has no git remote configured yet -- fill in REPO_URL if you've
    # pushed it somewhere, or mount Drive and point PROJECT_ROOT at a copy you
    # uploaded there. Leave REPO_URL empty to use the Drive-mount path.
    REPO_URL = ""
    PROJECT_DIRNAME = "tuna-ocr"

    if REPO_URL:
        if not Path(f"/content/{PROJECT_DIRNAME}").exists():
            get_ipython().system(f"git clone {REPO_URL} /content/{PROJECT_DIRNAME}")
        os.chdir(f"/content/{PROJECT_DIRNAME}/notebooks")
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        # Adjust this to wherever you uploaded the project under Drive.
        drive_path = Path(f"/content/drive/MyDrive/{PROJECT_DIRNAME}")
        assert drive_path.exists(), (
            f"expected the project at {drive_path} -- upload it there, or set "
            "REPO_URL above to clone from git instead"
        )
        os.chdir(drive_path / "notebooks")

    get_ipython().system("pip install -q -r ../data_gen/requirements.txt -r ../real_data/requirements.txt")

print("Colab:", IN_COLAB)

In [ ]:
import subprocess
import sys
import csv
from pathlib import Path
from IPython.display import display
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PROJECT_ROOT / "data_gen").is_dir(), f"expected data_gen/ under {PROJECT_ROOT}"

venv_python = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON = str(venv_python) if venv_python.exists() else sys.executable
print("project root:", PROJECT_ROOT)
print("python:", PYTHON)


def run(module: str, args: list[str]):
    """Run `python -m <module> <args>` from PROJECT_ROOT, streaming output live."""
    cmd = [PYTHON, "-m", module, *[str(a) for a in args]]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT)
    if proc.returncode != 0:
        raise RuntimeError(f"{module} exited with code {proc.returncode}")


def show_manifest(manifest_path: Path, n: int = 3):
    """Print row count and preview the first n rows of a manifest.tsv."""
    if not manifest_path.exists():
        print(f"(no manifest at {manifest_path})")
        return
    with manifest_path.open(encoding="utf-8") as f:
        rows = list(csv.reader(f, delimiter="\t"))
    header, body = rows[0], rows[1:]
    print(f"{manifest_path}: {len(body)} rows")
    print("\t".join(header))
    for row in body[:n]:
        print("\t".join(row))


def show_images(images_dir: Path, n: int = 3):
    """Display the first n images (sorted by name) found in a directory."""
    if not images_dir.exists():
        print(f"(no images dir at {images_dir})")
        return
    for path in sorted(images_dir.iterdir())[:n]:
        display(Image.open(path))

In [ ]:
CONFIG = {
    # -- synthetic (data_gen) --
    "lines_out_dir": PROJECT_ROOT / "dataset" / "lines",
    "lines_num_samples": 50,
    "pages_out_dir": PROJECT_ROOT / "dataset" / "pages",
    "pages_num_pages": 10,
    "seed": 0,
    # -- real (real_data) --
    "real_out_dir": PROJECT_ROOT / "real_data" / "samples",
    "real_sources": ["mrrtmob", "soyvitou_handwritten", "kiteaether"],  # or ["all"]
    "real_num_samples": 5,
    "real_shuffle_buffer": 0,  # keep 0 unless you have a fast connection -- see real_data/README.md
}
CONFIG

## Stage 1 -- synthetic line-recognition dataset (`data_gen.generate_lines`)

Image + transcript pairs for training the recognizer directly.

In [ ]:
run("data_gen.generate_lines", [
    "--out-dir", CONFIG["lines_out_dir"],
    "--num-samples", CONFIG["lines_num_samples"],
    "--seed", CONFIG["seed"],
])
show_manifest(CONFIG["lines_out_dir"] / "manifest.tsv")
show_images(CONFIG["lines_out_dir"] / "images")

## Stage 2 -- synthetic page + YOLO layout dataset (`data_gen.generate_pages`)

Full-page layouts with `dataset.yaml` for the layout detector, plus per-line
transcripts.

In [ ]:
run("data_gen.generate_pages", [
    "--out-dir", CONFIG["pages_out_dir"],
    "--num-pages", CONFIG["pages_num_pages"],
    "--seed", CONFIG["seed"],
])
transcripts_path = CONFIG["pages_out_dir"] / "line_transcripts.jsonl"
if transcripts_path.exists():
    with transcripts_path.open(encoding="utf-8") as f:
        n_lines = sum(1 for _ in f)
    print(f"{transcripts_path}: {n_lines} lines")
show_images(CONFIG["pages_out_dir"] / "images")

## Stage 3 -- synthetic PDF documents (`data_gen.generate_pdfs`)

Document-level variety (no bounding boxes -- PDF output, not raster). Unlike
the other two generators, this script takes **no CLI arguments** -- it has no
`argparse` at all, and always writes a fixed `NUM_DOCS` (currently 10, edit
the constant in `data_gen/generate_pdfs.py` to change it) PDFs to the
hardcoded `data_gen/ocr_dataset/`.

In [ ]:
run("data_gen.generate_pdfs", [])
pdfs_out_dir = PROJECT_ROOT / "data_gen" / "ocr_dataset"
sorted(pdfs_out_dir.glob("*.pdf"))

## Stage 4 -- real external data + overlapping chunks (`real_data.generate_external_chunks`)

Pulls streamed samples from the external HF datasets in
`real_data.config.EXTERNAL_DATASETS` and slices each line image into
fixed-width overlapping chunks for Conformer-encoder input windowing. Each
chunk shares its line's single whole-line transcript -- see
`real_data/README.md` for why (none of these datasets have per-character/word
boxes).

Network-dependent and can be slow per-row (see the shuffle-buffer note in
`real_data/README.md`); a failing source is skipped with a warning when
`CONFIG["real_sources"]` has more than one entry, since the CLI's `--source`
is invoked once per source below.

In [ ]:
for source in CONFIG["real_sources"]:
    out_dir = CONFIG["real_out_dir"] / source
    try:
        run("real_data.generate_external_chunks", [
            "--source", source,
            "--out-dir", out_dir,
            "--num-samples", CONFIG["real_num_samples"],
            "--seed", CONFIG["seed"],
            "--shuffle-buffer", CONFIG["real_shuffle_buffer"],
        ])
    except RuntimeError as e:
        print(f"warning: {source} failed: {e}")
        continue
    show_manifest(out_dir / "manifest.tsv")
    show_images(out_dir / "images")

## Summary

Row counts for every manifest produced by the stages run above.

In [ ]:
manifests = [
    CONFIG["lines_out_dir"] / "manifest.tsv",
    *[(CONFIG["real_out_dir"] / s / "manifest.tsv") for s in CONFIG["real_sources"]],
]
for m in manifests:
    if m.exists():
        with m.open(encoding="utf-8") as f:
            n = sum(1 for _ in f) - 1
        print(f"{m.relative_to(PROJECT_ROOT)}: {n} rows")
    else:
        print(f"{m.relative_to(PROJECT_ROOT)}: not generated yet")

pages_transcripts = CONFIG["pages_out_dir"] / "line_transcripts.jsonl"
if pages_transcripts.exists():
    with pages_transcripts.open(encoding="utf-8") as f:
        n = sum(1 for _ in f)
    print(f"{pages_transcripts.relative_to(PROJECT_ROOT)}: {n} lines")
else:
    print(f"{pages_transcripts.relative_to(PROJECT_ROOT)}: not generated yet")